# Validate extraction-quality improvements

Builds a small knowledge graph from a chunk of `wikipedia_test.txt` and measures the
anti-degeneracy guards that were just added:

- **No phantom `UNRESOLVED` nodes** (KnowledgeOrganizer now skips unresolved endpoints).
- **Low rogue-entity rate** (`compute_rogue_entity_stats`: UNRESOLVED / GENERIC / ORPHAN).
- **Low triple degeneracy** (`compute_degeneracy_rate`: SELF_REF / CONTAINMENT / GENERIC / TAUTOLOGY).
- **Graph connectivity** (`governed_kg.get_stats()['connectivity']`).

Also runs the offline Pydantic validation smoke test (no LLM) so you can confirm the
schema layer before spending any tokens.

> There is no automatic before/after here — the guards are preventive. The win is shown by
> the *absence* of phantom/degenerate structure on a fresh build. To get a true before/after,
> `git stash` these changes, rebuild, compare the printed numbers, then `git stash pop`.

In [ ]:
# === 0. Clone repo & checkout the exact branch (run this first) ===
# Clones if missing, then checks out + pulls the feature branch and installs
# the package INTO THIS KERNEL. No manual upload needed.
import os, sys

REPO_DIR = os.path.expanduser('~/Agentic-Graph-Memory')
BRANCH = 'feat/vector-index-and-qa-improvements'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/nmokaria27/Agentic-Graph-Memory.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull origin {BRANCH}

# IMPORTANT: install into THIS kernel's interpreter (sys.executable). Installing
# into a different conda env is why imports failed with ModuleNotFoundError.
!{sys.executable} -m pip install --no-build-isolation -e . -q
!{sys.executable} -m pip install pydantic datasets json_repair sentence_transformers scipy -q

REPO_ROOT = REPO_DIR
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Sanity check: confirm imports resolve in THIS kernel before continuing.
import importlib
for _m in ("pydantic", "multi_agent_kg", "multi_agent_kg.schemas_llm"):
    importlib.import_module(_m)
print('\n=== repo ready ===', REPO_ROOT)
print('branch :', BRANCH)
print('python :', sys.version.split()[0], '|', sys.executable)

import os, sys

# Reuse REPO_ROOT from the clone cell if it ran; otherwise assume this notebook
# is being run in-place from the repo's evaluation/ directory.
REPO_ROOT = globals().get("REPO_ROOT") or os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo:", REPO_ROOT)

# --- JupyterHub / vLLM (SSH tunnel) ---
# os.environ["LLM_BACKEND"] = "vllm"
# os.environ["VLLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
# os.environ["LLM_DEFAULT_MODEL"] = "your-served-model"

# --- Colab / Fireworks (OpenAI-compatible) ---
# Read the key from a secret/env — never hard-code it in the notebook.
# os.environ["LLM_BACKEND"] = "openai"
# os.environ["OPENAI_API_KEY"] = os.environ["FIREWORKS_API_KEY"]
# os.environ["OPENAI_BASE_URL"] = "https://api.fireworks.ai/inference/v1"
# os.environ["LLM_DEFAULT_MODEL"] = "accounts/fireworks/models/deepseek-v3"

print("LLM_BACKEND =", os.getenv("LLM_BACKEND", "(unset → ollama default)"))

In [ ]:
import os, sys

# Point this at your repo root (Colab: usually /content/Agent-Graph-Memory).
REPO_ROOT = os.path.abspath("..")  # this notebook lives in evaluation/
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo:", REPO_ROOT)

# --- JupyterHub / vLLM (SSH tunnel) ---
# os.environ["LLM_BACKEND"] = "vllm"
# os.environ["VLLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
# os.environ["LLM_DEFAULT_MODEL"] = "your-served-model"

# --- Colab / Fireworks (OpenAI-compatible) ---
# os.environ["LLM_BACKEND"] = "openai"
# os.environ["OPENAI_API_KEY"] = os.environ["FIREWORKS_API_KEY"]
# os.environ["OPENAI_BASE_URL"] = "https://api.fireworks.ai/inference/v1"
# os.environ["LLM_DEFAULT_MODEL"] = "accounts/fireworks/models/deepseek-v3"

print("LLM_BACKEND =", os.getenv("LLM_BACKEND", "(unset → ollama default)"))

## 2. Offline schema smoke (no LLM, instant)

Proves the Pydantic boundary is lossless + crash-safe before any token is spent.

In [ ]:
from scripts.smoke_test_pydantic_validation import part_a_offline
part_a_offline()

SOURCE = os.path.join(REPO_ROOT, "wikipedia_test.txt")
CHUNK_CHARS = 1500  # keep small for a fast/cheap validation run

if not os.path.exists(SOURCE):
    raise FileNotFoundError(
        f"{SOURCE} not found. Re-run cell 0 (clone/pull) — wikipedia_test.txt is "
        "committed to the repo, so a fresh pull brings it in. If you opened this "
        "notebook in-place, make sure REPO_ROOT points at the repo root."
    )

with open(SOURCE, "r", encoding="utf-8") as fh:
    full_text = fh.read()
chunk = full_text[:CHUNK_CHARS]
print(f"source chars: {len(full_text):,}  |  using first {len(chunk):,}\n")
print(chunk[:600], "...")

In [ ]:
SOURCE = os.path.join(REPO_ROOT, "wikipedia_test.txt")
CHUNK_CHARS = 1500  # keep small for a fast/cheap validation run

with open(SOURCE, "r", encoding="utf-8") as fh:
    full_text = fh.read()
chunk = full_text[:CHUNK_CHARS]
print(f"source chars: {len(full_text):,}  |  using first {len(chunk):,}\n")
print(chunk[:600], "...")

## 4. Build a small governed KG

Runs the real extraction pipeline (entities → relations → organizer admission). Self-consistency,
deliberation and cross-document are disabled to keep it quick. This is the path that exercises the
UNRESOLVED-skip and degenerate-endpoint guards.

In [ ]:
from multi_agent_kg.core.deliberative_orchestrator import DeliberativeOrchestrator
from multi_agent_kg.core.knowledge_graph import KnowledgeGraph
from multi_agent_kg.core.governed_kg import GovernedKnowledgeGraph
from multi_agent_kg.core.config import LLMConfig

orchestrator = DeliberativeOrchestrator(
    llm_config=LLMConfig(temperature=0.2, max_tokens=4096),
    knowledge_graph=KnowledgeGraph(),
    governed_kg=GovernedKnowledgeGraph(governance_mode="audit_only"),
    quality_threshold=0.35,
    max_refinement_iterations=1,
    enable_self_consistency=False,
    enable_deliberation=False,
    enable_cross_document=False,
    chunk_size=1200,
)

_ = orchestrator.process_corpus([
    {"id": "wiki_chunk", "text": chunk, "metadata": {"source": "wikipedia_test.txt"}}
])

gk = orchestrator.governed_kg
kg_dict = gk.knowledge_graph.to_dict() if hasattr(gk, "knowledge_graph") else gk._kg.to_dict()
entities = kg_dict["entities"]
triples = kg_dict["triples"]
print(f"\nBUILD DONE — entities: {len(entities)}  triples: {len(triples)}")

## 5. Score with the new quality metrics

In [ ]:
import json
from evaluation.evaluate_kg import compute_degeneracy_rate, compute_rogue_entity_stats

degen = compute_degeneracy_rate(triples)
rogue = compute_rogue_entity_stats(entities, triples)
conn = gk.get_stats().get("connectivity", {})

print("=== TRIPLE DEGENERACY ===")
print(f"  rate: {degen['degeneracy_rate']:.4f}  ({degen['degenerate_count']}/{len(triples)})")
print(f"  by_type: {degen['by_type']}")

print("\n=== ROGUE ENTITIES ===")
print(f"  rate: {rogue['rogue_entity_rate']:.4f}  ({rogue['rogue_entity_count']}/{len(entities)})")
print(f"  by_type: {rogue['by_type']}")

print("\n=== CONNECTIVITY ===")
print(f"  {json.dumps(conn, indent=2)}")

## 6. Direct checks on the guards

In [ ]:
# (a) No phantom UNRESOLVED / auto_created nodes should exist anymore.
unresolved = [
    e for e in entities
    if str(e.get("type", "")).upper() == "UNRESOLVED"
    or (e.get("metadata", {}) or {}).get("auto_created")
]
print(f"UNRESOLVED / auto_created nodes: {len(unresolved)}  (expect 0)")
for e in unresolved[:5]:
    print("   ⚠", e.get("id"), e.get("type"))

# (b) No surviving degenerate (self-ref / containment-loop) triples.
print(f"\nDegenerate triples surviving filters: {degen['degenerate_count']}  (expect ~0)")
for flag, exs in degen.get("examples", {}).items():
    for t in exs[:2]:
        print(f"   ⚠ {flag}: ({t.get('subject')}) -[{t.get('relation')}]-> ({t.get('object')})")

# (c) Sample of what was actually extracted.
print("\nSample entities:", [e.get("id") for e in entities[:12]])
print("\nSample triples:")
for t in triples[:12]:
    print(f"   ({t.get('subject')}) -[{t.get('relation')}]-> ({t.get('object')})  conf={t.get('confidence')}")

## How to read this

| Signal | Healthy | Means |
|---|---|---|
| `UNRESOLVED / auto_created nodes` | **0** | organizer no longer invents phantom endpoints |
| `rogue_entity_rate` | low; `by_type` UNRESOLVED ≈ 0 | clean entity set; remaining ORPHAN are just unlinked, not invalid |
| `degeneracy_rate` | ≈ 0 | self-ref / containment-loop triples filtered before governance |
| `connectivity.connectivity_ratio` | higher = better | larger share of entities in the main connected component |

**True before/after:** `git stash` → re-run cells 4–5 → note numbers → `git stash pop` → re-run → compare.
Expect UNRESOLVED nodes and degenerate triples to drop to ~0 with the guards on.